# Lab Tuần 4 — Xử lý dữ liệu theo lô với PySpark

Yêu cầu trước khi bắt đầu: đã cài `pyspark` và JDK 11/17, và thư mục `data/`
nằm cùng cấp với thư mục chứa notebook này.

```bash
pip install pyspark
```
---

## Bài 0 — Khởi động

Mục tiêu: dựng `SparkSession`, kiểm tra môi trường và mở được Spark UI.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import json, time

spark = (SparkSession.builder
         .appName("Lab-Tuan4")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

DATA = "/home/lucas/Desktop/Github/BigData/homework/week6/BaiTap _Mapreduce/data"
print("Spark", spark.version, "| so core:", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 14:32:36 WARN Utils: Your hostname, lucas-Dell-G15-5525, resolves to a loopback address: 127.0.1.1; using 192.168.2.28 instead (on interface wlp4s0)
26/09/21 14:32:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/lucas/Desktop/Github/BigData/homework/bigdata-env/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/21 14:32:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | so core: 12
Spark UI: http://192.168.2.28:4040


### Câu hỏi

Cụm đang chạy ở chế độ Cluster Manager nào? `local[*]` tương ứng bao nhiêu Executor?

**Trả lời:** Chế độ Local mode, không sử dụng Cluster Manager bên ngoài. `local[*]` tạo 1 Executor duy nhất với số thread bằng tổng số core khả dụng trên máy.

## Bài 1 — WordCount và vai trò của Combiner

Dữ liệu: `document.txt` (300 dòng văn bản).

Viết WordCount bằng **hai cách** rồi so sánh chi phí:
- **1a** dùng `reduceByKey`
- **1b** dùng `groupByKey` + `mapValues`

In [2]:
lines = sc.textFile(f"{DATA}/access.log")

wc_reduce = (lines.flatMap(lambda l: l.split())
                  .map(lambda w: w.lower())
                  .map(lambda w: (w, 1))
                  .reduceByKey(lambda a, b: a + b))

wc_group = (lines.flatMap(lambda l: l.split())
                 .map(lambda w: w.lower())
                 .map(lambda w: (w, 1))
                 .groupByKey()
                 .mapValues(lambda vals: sum(vals)))

print("Top 5:", sorted(wc_reduce.collect(), key=lambda x: -x[1])[:5])
print("Hai cach cho cung ket qua:",
      sorted(wc_reduce.collect()) == sorted(wc_group.collect()))

Top 5: [('200', 5253), ('info', 5253), ('/api/search', 3296), ('post', 3068), ('get', 2932)]


Hai cach cho cung ket qua: True


### Câu hỏi

Hai cách cho cùng kết quả. Vậy khác nhau ở đâu, và vì sao `reduceByKey` lại rẻ hơn?

**Trả lời:** `reduceByKey` thực hiện combine cục bộ trên từng partition trước khi shuffle, nên lượng dữ liệu truyền qua mạng ít hơn nhiều. `groupByKey` chuyển toàn bộ cặp (key, value) qua shuffle rồi mới gộp nhóm, gây tốn băng thông và bộ nhớ.

## Bài 2 — Phép trung bình

Dữ liệu: `temperatures.csv` — cột `station_id, year, temperature`.

Tính nhiệt độ trung bình toàn quốc bằng **hai cách**:
- **2a — cách sai**: tính trung bình từng trạm, rồi lấy trung bình của các trung bình
- **2b — cách đúng**: dùng `aggregateByKey` giữ cặp `(tổng, số lượng)`, chia ở bước cuối

In [3]:
raw = sc.textFile(f"{DATA}/temperatures.csv")
header = raw.first()
pairs = (raw.filter(lambda l: l != header)
            .map(lambda l: l.split(","))
            .map(lambda c: (c[0], float(c[2]))))
print("So ban ghi:", pairs.count())

tb_tram = pairs.mapValues(lambda t: (t, 1)).reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda x: x[0]/x[1])
all_vals = tb_tram.values().collect()
tb_sai = sum(all_vals) / len(all_vals)

tb_dung = (pairs.aggregateByKey((0.0, 0),
                                lambda acc, v: (acc[0] + v, acc[1] + 1),
                                lambda a, b: (a[0] + b[0], a[1] + b[1]))
                .values()
                .reduce(lambda a, b: (a[0] + b[0], a[1] + b[1])))
tb_dung = tb_dung[0] / tb_dung[1]

print("Trung binh SAI :", round(tb_sai, 2))
print("Trung binh DUNG:", round(tb_dung, 2))
print("Chenh lech     :", round(abs(tb_dung - tb_sai), 2), "do C")

So ban ghi: 6001


Trung binh SAI : 22.16
Trung binh DUNG: 28.08
Chenh lech     : 5.92 do C


> **Kết quả mong đợi:** Trung binh SAI ≈ 22.16 | DUNG ≈ 28.08 | chênh ≈ 5.92 °C

### Câu hỏi

Vì sao phép trung bình không dùng trực tiếp làm Combiner được, trong khi SUM, MAX, COUNT thì được?

**Trả lời:** Trung bình không có tính kết hợp: avg(avg(A), avg(B)) ≠ avg(A ∪ B) khi kích thước hai tập khác nhau. SUM, MAX, COUNT đều có tính kết hợp nên gộp cục bộ cho kết quả đúng.

## Bài 3 — Accumulator và dữ liệu bẩn

Dữ liệu: `events.jsonl` — JSON Lines, có một số dòng cố tình hỏng.

Dùng `Accumulator` đếm số bản ghi không parse được ngay trong lúc xử lý.

In [4]:
loi_parse = sc.accumulator(0)

def parse_an_toan(dong):
    try:
        return json.loads(dong)
    except Exception:
        loi_parse.add(1)
        return None

ban_ghi = sc.textFile(f"{DATA}/events.jsonl").map(parse_an_toan).filter(lambda x: x is not None)

print("So ban ghi hop le:", ban_ghi.count())
print("So ban ghi loi   :", loi_parse.value)

So ban ghi hop le: 2844
So ban ghi loi   : 156


> **Kết quả mong đợi:** 2844 bản ghi hợp lệ, 156 bản ghi lỗi

### Câu hỏi

Nếu đọc `loi_parse.value` **trước** khi gọi `count()` thì được bao nhiêu? Và vì sao con số này có thể sai khi chạy trên cụm thật?

**Trả lời:** Trước khi gọi `count()`, giá trị là 0 vì chưa có action nào kích hoạt tính toán. Trên cụm thật, executor gửi accumulator về driver không đồng bộ, nếu task bị retry do speculatie execution hoặc failure thì giá trị có thể bị đếm trùng.

## Bài 4 — DataFrame, Join và left_anti

Dữ liệu: `customers.csv` (600 khách) và `orders.csv`.

Tìm khách hàng **chưa từng đặt đơn nào** bằng hai cách.

In [5]:
kh = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
dh = spark.read.csv(f"{DATA}/orders.csv",    header=True, inferSchema=True)

key = dh.columns[1]                      # cot join trong orders (customer_id)
dh2 = dh.withColumnRenamed(key, key + "_dh")   # doi ten de tranh ambiguous

chua_mua_a = kh.join(dh2, kh[key] == dh2[key + "_dh"], "left_anti")

chua_mua_b = (kh.join(dh2, kh[key] == dh2[key + "_dh"], "left")
                .filter(F.col(key + "_dh").isNull())
                .select(kh.columns))

print("left_anti       :", chua_mua_a.count())
print("left join + null:", chua_mua_b.count())
chua_mua_a.explain()

left_anti       : 151


left join + null: 151
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [customer_id#17], [customer_id_dh#44], LeftAnti, BuildRight, false, false
   :- FileScan csv [customer_id#17,name#18,city#19,segment#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/lucas/Desktop/Github/BigData/homework/week6/BaiTap _Mapredu..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,name:string,city:string,segment:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=292]
      +- Project [customer_id#39 AS customer_id_dh#44]
         +- Filter isnotnull(customer_id#39)
            +- FileScan csv [customer_id#39] Batched: false, DataFilters: [isnotnull(customer_id#39)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/lucas/Desktop/Github/BigData/homework/week6/BaiTap _Mapredu..., PartitionFilters: [], PushedFilters: [IsNotNul

> **Kết quả mong đợi:** Cả hai cách đều ra 151 khách hàng

### Câu hỏi

Hai cách cho cùng kết quả. Cách nào rẻ hơn và vì sao?

**Trả lời:** `left_anti` rẻ hơn vì Catalyst chỉ cần kiểm tra sự tồn tại của key bên phải, không cần mang theo cột từ bảng orders hay tạo bản ghi null rồi lọc bỏ.

## Bài 5 — Data skew và Custom Partitioner

Dữ liệu: `access.log` và `big_file.csv`.

**5a** Đếm số lượt truy cập theo `endpoint`.
**5b** Phân vùng `big_file.csv` theo domain email bằng Custom Partitioner.

In [6]:
logs = sc.textFile(f"{DATA}/access.log")
phan_bo = (logs.map(lambda l: l.split()[3])
               .map(lambda ep: (ep, 1))
               .reduceByKey(lambda a, b: a + b)
               .sortBy(lambda x: -x[1]))
for ep, n in phan_bo.collect():
    print("%-16s %5d" % (ep, n))

def theo_domain(key):
    domain = key.split("@")[1] if "@" in key else "unknown"
    return hash(domain) % 5

df = spark.read.csv(f"{DATA}/big_file.csv", header=True, inferSchema=True)
rdd = df.rdd.map(lambda r: (r["email"], r["amount"]))
kich_thuoc = (rdd.partitionBy(5, theo_domain)
                 .mapPartitionsWithIndex(lambda i, it: [(i, sum(1 for _ in it))])
                 .collect())
print("Kich thuoc tung partition:", kich_thuoc)

/api/search       3296
/api/product      1050
/api/cart          709
/api/login         487
/api/checkout      328
/api/admin         130


Kich thuoc tung partition: [(0, 0), (1, 40064), (2, 0), (3, 39831), (4, 20105)]


> **Kết quả mong đợi:** `/api/search` chiếm ~55%, gấp ~25 lần endpoint nhỏ nhất

### Câu hỏi

Trong 5a, nếu mỗi endpoint đi về một Reducer riêng thì điều gì xảy ra với tổng thời gian chạy?

**Trả lời:** Reducer nhận `/api/search` sẽ xử lý ~55% dữ liệu, trở thành bottleneck. Tổng thời gian chạy bị chi phối bởi reducer chậm nhất thay vì trung bình cộng.

## Bài 6 — Đo tác động của cache()

In [7]:
big = sc.textFile(f"{DATA}/big_file.csv").filter(lambda l: "gmail.com" in l)

t1 = time.time()
big.count()
t2 = time.time()
big.count()
t3 = time.time()
print("Chua cache - lan 1: %.3fs" % (t2 - t1))
print("Chua cache - lan 2: %.3fs" % (t3 - t2))

big.cache()
big.count()

t4 = time.time()
big.count()
t5 = time.time()
print("Da cache: %.3fs" % (t5 - t4))

Chua cache - lan 1: 0.078s
Chua cache - lan 2: 0.054s


Da cache: 0.021s


### Câu hỏi

Khi chưa cache, vì sao lần `count()` thứ hai vẫn tốn gần bằng lần đầu?

**Trả lời:** RDD có tính lazy và không lưu kết quả giữa các action. Mỗi lần gọi `count()`, Spark đọc lại dữ liệu từ nguồn và thực hiện lại toàn bộ lineage từ đầu.

---
## Kết thúc

In [8]:
spark.stop()